In [0]:
%pip install databricks-labs-dqx

In [0]:
%restart_python

## Generate check yml

In [0]:
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.config import InputConfig

ws = WorkspaceClient()
profiler = DQProfiler(ws)

summary_stats, profiles = profiler.profile_table(
  input_config=InputConfig(location="biap_dev.bronze.bronze_nyc_taxi_trips")
)

print("Summary Statistics:", summary_stats)
print("Generated Profiles:", profiles)

In [0]:
import yaml
generator = DQGenerator(ws)
generated_checks = generator.generate_dq_rules(profiles)
print(generated_checks)

yaml_checks = yaml.safe_dump(generated_checks, default_flow_style=False)
print(yaml_checks)

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.config import WorkspaceFileChecksStorageConfig
dq_engine = DQEngine(ws)
checks: list[dict] = dq_engine.load_checks(config=WorkspaceFileChecksStorageConfig(location="/Workspace/Users/watchaaq@ais.co.th/nyc-taxi-pipeline/checks/silver_nyc_taxi_checks.yml"))

In [0]:
from databricks.labs.dqx.engine import DQEngine
dq_engine = DQEngine(ws)
dq_engine.save_checks(generated_checks, config=FileChecksStorageConfig(location="bronze_checks.yml"))

## Load and apply

In [0]:

from databricks.labs.dqx.config import WorkspaceFileChecksStorageConfig

from databricks.sdk import WorkspaceClient

checks: list[dict] = dq_engine.load_checks(config=WorkspaceFileChecksStorageConfig(location="/Workspace/Users/watchaaq@ais.co.th/nyc-taxi-pipeline/explorations/bronze_nyc_taxi_checks.yml"))
print(checks)

In [0]:
# get vaild/ invalid


dq_engine_check = DQEngine(WorkspaceClient())
valid_df, invalid_df = dq_engine_check.apply_checks_by_metadata_and_split(input_df, checks)
display("Vaild_df")
display(valid_df)
display("Invaild_df")
display(invalid_df)


## Metrics

In [0]:
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.metrics_observer import DQMetricsObserver
from databricks.sdk import WorkspaceClient

# Create observer
observer = DQMetricsObserver(name="dq_metrics")

# Create the engine with the optional observer
engine = DQEngine(WorkspaceClient(), observer=observer)

# Apply checks and get metrics
checked_df, observation = engine.apply_checks_by_metadata(input_df, checks)

# Apply checks, split and get metrics
#valid_df, quarantine_df, observation = engine.apply_checks_by_metadata_and_split(df, checks)

# Trigger an action to populate metrics (e.g., count, save to a table).
# Without triggering an action, metrics will not be populated, and accessing them will result in a stall.
row_count = checked_df.count()

# Access metrics
metrics = observation.get
print(f"Input row count: {metrics['input_row_count']}")
print(f"Error row count: {metrics['error_row_count']}")
print(f"Warning row count: {metrics['warning_row_count']}")
print(f"Valid row count: {metrics['valid_row_count']}")
print(f"Check metrics: {metrics['check_metrics']}")  # per-check error/warning counts as JSON